In [1]:
## READS MNQ-TICK from HGT-Export SC chartbook

In [2]:
import pandas as pd
import numpy as np
import datetime as dt 
from pathlib import Path
import time

In [3]:
# Configuration: Style preferences
#plt.style.use('ggplot') # Good default for readability
pd.set_option("display.width", 400)      # total characters per line
pd.set_option("display.max_columns", 30) # prevent wrapping by limiting columns
pd.set_option("display.max_rows", 1000)

In [4]:
import os
os.getcwd()

'/home/vm/pt/hgt-rl/mnq-tick/nq'

In [5]:
inFile = f'/mnt/d/SierraChart/data/EXPORT/NQ-OHLC-3SEC.csv'
outFile= f'data/nq-ohlc-raw-3sec.pqt'

print(inFile, outFile,)

/mnt/d/SierraChart/data/EXPORT/NQ-OHLC-3SEC.csv data/nq-ohlc-raw-3sec.pqt


In [6]:
df = pd.read_csv(inFile)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 10916647 entries, 0 to 10916646
Data columns (total 13 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Date       str    
 1   Time       str    
 2   Open       float64
 3   High       float64
 4   Low        float64
 5   Last       float64
 6   Volume     int64  
 7   #ofTrades  int64  
 8   OHLCAvg    float64
 9   HLCAvg     float64
 10  HLAvg      float64
 11  BidVolume  int64  
 12  AskVolume  int64  
dtypes: float64(7), int64(4), str(2)
memory usage: 1.3 GB
None
       Date             Time      Open      High       Low      Last  Volume  #ofTrades   OHLCAvg    HLCAvg     HLAvg  BidVolume  AskVolume
0  2022-1-3  08:00:00.000000  16430.75  16430.75  16429.75  16429.75      23         22  16430.25  16430.08  16430.25         17          6
1  2022-1-3  08:00:03.000000  16430.25  16430.50  16428.75  16429.25      18         17  16429.69  16429.50  16429.63         11          7
2  2022-1-3  08:00:06.000000  16429.00  16429.75  164

In [7]:
# expand stupid SC date like 2026-7-8 to 2026-07-08
parts = df['Date'].astype(str).str.split('-', expand=True)

year  = parts[0]
month = parts[1].str.zfill(2)
day   = parts[2].str.zfill(2)

df['Date_norm'] = year + '-' + month + '-' + day

# join date and time and convert to wall clock datetime64
df['timestamp'] = pd.to_datetime(
    df['Date_norm'] + ' ' + df['Time'].astype(str),
    utc=False,            # keep as wall clock, no timezone
    errors='raise'        # or 'coerce' if you want bad rows as NaT
)

# remove temp columns
df = df.drop(columns=['Date', 'Date_norm', 'Time'])

# make timestamp first column
col = df.pop('timestamp')
df.insert(0, 'timestamp', col)

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 10916647 entries, 0 to 10916646
Data columns (total 12 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   Open       float64       
 2   High       float64       
 3   Low        float64       
 4   Last       float64       
 5   Volume     int64         
 6   #ofTrades  int64         
 7   OHLCAvg    float64       
 8   HLCAvg     float64       
 9   HLAvg      float64       
 10  BidVolume  int64         
 11  AskVolume  int64         
dtypes: datetime64[us](1), float64(7), int64(4)
memory usage: 999.4 MB
None
            timestamp      Open      High       Low      Last  Volume  #ofTrades   OHLCAvg    HLCAvg     HLAvg  BidVolume  AskVolume
0 2022-01-03 08:00:00  16430.75  16430.75  16429.75  16429.75      23         22  16430.25  16430.08  16430.25         17          6
1 2022-01-03 08:00:03  16430.25  16430.50  16428.75  16429.25      18         17  16429.69  16429.50  16429.63         1

In [8]:
df.drop(columns=['Volume','#ofTrades','BidVolume','AskVolume'], inplace=True) 
#df.drop(columns=['Volume','NumberOfTrades','BidVolume','AskVolume'], inplace=True) 

print(df.info())
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 10916647 entries, 0 to 10916646
Data columns (total 8 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   Open       float64       
 2   High       float64       
 3   Low        float64       
 4   Last       float64       
 5   OHLCAvg    float64       
 6   HLCAvg     float64       
 7   HLAvg      float64       
dtypes: datetime64[us](1), float64(7)
memory usage: 666.3 MB
None
            timestamp      Open      High       Low      Last   OHLCAvg    HLCAvg     HLAvg
0 2022-01-03 08:00:00  16430.75  16430.75  16429.75  16429.75  16430.25  16430.08  16430.25
1 2022-01-03 08:00:03  16430.25  16430.50  16428.75  16429.25  16429.69  16429.50  16429.63
2 2022-01-03 08:00:06  16429.00  16429.75  16429.00  16429.50  16429.31  16429.42  16429.38
3 2022-01-03 08:00:09  16428.75  16430.25  16428.75  16429.25  16429.25  16429.42  16429.50
4 2022-01-03 08:00:12  16429.75  16429.75  16429.75  16429.75

In [9]:
print(f'monotonic: {df["timestamp"].is_monotonic_increasing}')

df['date'] = df['timestamp'].dt.normalize()

filtered_days = [
   '2022-01-17', '2022-07-04', '2022-11-24', '2023-01-16',
   '2023-02-20', '2023-04-07', '2023-05-29', '2023-06-19',
   '2023-07-04', '2023-09-04', '2023-11-23', '2024-02-19',
   '2024-05-27', '2024-06-19', '2024-07-04', '2024-11-28',
   '2025-01-09', '2025-07-04', '2025-09-01', '2025-11-27',
   '2026-04-03', '2026-07-09',               '2025-11-28', 
]

# before 1168
days = df["date"]
all_days = days.drop_duplicates().sort_values()
print("Unique days before:", len(all_days))

df = df[
    ~df["timestamp"].dt.normalize().isin(pd.to_datetime(filtered_days))
]

# after -23 = 1145
days = df["date"]
all_days = days.drop_duplicates().sort_values()
print(" Unique days after:", len(all_days))

#

print(df.info())
print(df.head())

monotonic: True
Unique days before: 1169
 Unique days after: 1146
<class 'pandas.DataFrame'>
Index: 10819394 entries, 0 to 10916646
Data columns (total 9 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[us]
 1   Open       float64       
 2   High       float64       
 3   Low        float64       
 4   Last       float64       
 5   OHLCAvg    float64       
 6   HLCAvg     float64       
 7   HLAvg      float64       
 8   date       datetime64[us]
dtypes: datetime64[us](2), float64(7)
memory usage: 825.5 MB
None
            timestamp      Open      High       Low      Last   OHLCAvg    HLCAvg     HLAvg       date
0 2022-01-03 08:00:00  16430.75  16430.75  16429.75  16429.75  16430.25  16430.08  16430.25 2022-01-03
1 2022-01-03 08:00:03  16430.25  16430.50  16428.75  16429.25  16429.69  16429.50  16429.63 2022-01-03
2 2022-01-03 08:00:06  16429.00  16429.75  16429.00  16429.50  16429.31  16429.42  16429.38 2022-01-03
3 2022-01-03 08:0

In [10]:
df.to_parquet(outFile, index=False)

In [11]:
print(df["timestamp"].dt.time.min())     
x = 1/0

08:00:00


ZeroDivisionError: division by zero

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt 
from pathlib import Path
import time

x= f'data/mnq-ohlc-raw-3sec.pqt'
df = pd.read_parquet(x)

# Unique calendar days in the full DataFrame
days = df["timestamp"].dt.normalize()
all_days = days.drop_duplicates().sort_values()

print("Unique days:", len(all_days))

# Keep rows from 09:00 inclusive to 12:00 exclusive
mask = df["timestamp"].dt.hour.between(9, 11)
df = df.loc[mask]

# Number of matching rows per day, including zero for days without matches
rows_per_day = (
    df#.loc[mask]
      .groupby(df.loc[mask, "timestamp"].dt.normalize())
      .size()
      .reindex(all_days, fill_value=0)
)

# For each row count, show how many days had that count
count_distribution = rows_per_day.value_counts().sort_index()

print("\nRows per day:")
print(rows_per_day)

print("\nRow count -> number of days:")
print(count_distribution.rename_axis("row_count").rename("number_of_days"))

In [ ]:
N = 3584            

days_with_n_rows = rows_per_day[rows_per_day == N].index

print(f"Days with row_count = {N}:")
print(days_with_n_rows)

In [ ]:
N = 3500            

days_with_n_rows = rows_per_day[rows_per_day < N].index

print(f"Days with row_count = {N}:")
print(days_with_n_rows)

In [ ]:
D = "2022-01-17"

day_rows = (
    df.loc[df["timestamp"].dt.date == pd.Timestamp(D).date()]
      .sort_values("timestamp")
      .copy()
)

day_rows["next_timestamp"] = day_rows["timestamp"].shift(-1)
day_rows["gap"] = day_rows["next_timestamp"] - day_rows["timestamp"]

gaps = day_rows.loc[
    day_rows["gap"] > pd.Timedelta(seconds=3),
    ["timestamp", "next_timestamp", "gap"]
].copy()

gaps["missing_row_count"] = (
    gaps["gap"].div(pd.Timedelta(seconds=3)).astype(int) - 1
)

print(gaps)